# Playfit Intelligence Lab — 06: LambdaRank & DuckDB Analytics

Dos temas avanzados:

1. **LightGBM LambdaRank**: Modelo de ranking basado en gradient boosting comparado contra el híbrido
2. **DuckDB Analytics**: Motor OLAP embebido para consultas analíticas sobre Parquet

## 1. LightGBM LambdaRank

Entrenamos un `LGBMRanker` con `objective='lambdarank'` y comparamos NDCG contra el modelo híbrido.

In [ ]:
import sys; sys.path.insert(0, '..')
import warnings; warnings.filterwarnings('ignore')

import numpy as np
import lightgbm
print(f"LightGBM version: {lightgbm.__version__}")

from src.models.lambdarank import LambdaRankRecommender
from src.features.game_features import build_feature_matrix, compute_popularity_score, compute_richness_score
from src.models.content_based import build_content_model
from src.models.hybrid import HybridRecommender
from src.evaluation.metrics import ndcg_at_k

fm = compute_richness_score(compute_popularity_score(build_feature_matrix()))
cm = build_content_model(fm, n_components=100)

print("Training LambdaRank (this may take a few minutes)...")
lr = LambdaRankRecommender(n_estimators=100, learning_rate=0.1)
lr.fit(fm)
ndcg = lr.evaluate_ndcg(fm)

print("\nLambdaRank NDCG results:")
for k, v in ndcg.items():
    print(f"  {k}: {v:.4f}")

In [ ]:
# Compare with hybrid model
rec = HybridRecommender(alpha=0.5, beta=0.4, gamma=0.1)
rec.fit(fm, cm)

test_profiles = [
    {"name": "Casual", "tags": ["casual", "puzzle", "family"]},
    {"name": "Story", "tags": ["story_rich", "narrative", "adventure"]},
    {"name": "Hardcore", "tags": ["challenging", "action", "rpg"]},
]

hybrid_ndcgs = []
for prof in test_profiles:
    recs = rec.recommend_for_profile(prof["tags"], k=20)
    tag_cols = [c for c in prof["tags"] if c in fm.columns]
    relevant = set()
    for row in fm.to_dicts():
        if any(row.get(t, 0) == 1 for t in tag_cols):
            relevant.add(row["game_id"])
    ndcg_val = ndcg_at_k([r["game_id"] for r in recs], relevant, 10)
    hybrid_ndcgs.append(ndcg_val)

print("\nModel Comparison (NDCG@10):")
print(f"  LambdaRank avg: {np.mean(list(ndcg.values())):.4f}")
print(f"  Hybrid avg:     {np.mean(hybrid_ndcgs):.4f}")

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8, 5))
models = ['LambdaRank', 'Hybrid (α=0.5)']
scores = [np.mean(list(ndcg.values())), np.mean(hybrid_ndcgs)]
ax.bar(models, scores, color=['#9b59b6', '#3498db'], width=0.4)
ax.set_ylabel('Mean NDCG@10')
ax.set_title('LambdaRank vs Hybrid Recommender')
for i, s in enumerate(scores):
    ax.text(i, s + 0.005, f'{s:.4f}', ha='center', fontsize=12)
plt.tight_layout()
plt.savefig('../reports/figures/lambdarank_vs_hybrid.png', dpi=150, bbox_inches='tight')
plt.show()

## 2. DuckDB Analytics

DuckDB permite consultas SQL directamente sobre archivos Parquet, ideal para análisis exploratorio sin cargar datos en memoria.

In [ ]:
from src.data.analytics_duckdb import DuckDBAnalytics
da = DuckDBAnalytics()

print("Games by year (sample):")
print(da.games_by_year().sort("release_year", descending=True).head(10))

print("\nTop platforms:")
print(da.top_platforms(10))

print("\nAverage confidence by genre:")
print(da.avg_confidence_by_genre().head(10))

print("\nCatalog coverage (single query):")
print(da.coverage_pivot())

print("\nDuplicate groups by status:")
print(da.duplicate_summary())

### Window Functions y Approximate Counts

In [ ]:
print("\nTop games per platform (window function):")
print(da.ranked_games_by_platform().head(20))

print("\nApproximate tag diversity (approx vs exact):")
print(da.approximate_tag_diversity().head(10))

da.close()

### Performance: DuckDB vs Polars vs Pandas

In [ ]:

import time
import polars as pl
import pandas as pd

_ = da.query("SELECT count(*) FROM read_parquet('data/raw/game_tags.parquet')")

start = time.time()
for _ in range(5):
    q = "SELECT tag_id, count(*) as cnt FROM read_parquet('data/raw/game_tags.parquet') GROUP BY tag_id ORDER BY cnt DESC"
    r = da.query(q)
duck_time = (time.time() - start) / 5

pt = pl.read_parquet("data/raw/game_tags.parquet")
start = time.time()
for _ in range(5):
    r = pt.group_by("tag_id").len().sort("len", descending=True)
polars_time = (time.time() - start) / 5

pd_df = pd.read_parquet("data/raw/game_tags.parquet")
start = time.time()
for _ in range(5):
    r = pd_df.groupby("tag_id").size().sort_values(ascending=False)
pandas_time = (time.time() - start) / 5

print(f"\nPerformance: GROUP BY tag_id (avg of 5 runs)")
print(f"  DuckDB:  {duck_time*1000:.1f}ms")
print(f"  Polars:  {polars_time*1000:.1f}ms")
print(f"  Pandas:  {pandas_time*1000:.1f}ms")


## 3. Resumen

| Técnica | Librería | Resultado |
|---------|----------|-----------|
| LambdaRank | LightGBM 4.6 | Modelo de ranking competitivo con baseline |
| SQL Analytics | DuckDB 1.5 | Consultas directo sobre Parquet, sin carga |
| Window Functions | DuckDB SQL | `ROW_NUMBER() OVER (PARTITION BY)` |
| Approximate Count | DuckDB `approx_count_distinct()` | ~rápido para cardinalidad de tags |